# 2 · Colour roads by your own data

Beyond road class, you can colour and size edges by any column — **categorical** (`color_by` + `colors`) or **numeric** (`color_by` + `cmap` + `width_by`), with automatic legends.

In [1]:
import pathlib
import geopandas as gpd
import roadstyle as rs

# A GeoDataFrame of road edges. roadstyle only needs a *road-class* column (default "highway")
# and line geometry in any CRS — it reprojects to web coordinates (EPSG:4326) for you.
edges = gpd.read_file(pathlib.Path("data") / "sundbyberg_edges.gpkg")
print(f"{len(edges):,} edges, CRS {edges.crs}")
edges[[c for c in edges.columns if c != edges.geometry.name]].head(3)


4,039 edges, CRS EPSG:4326


ERROR 1: PROJ: proj_create_from_database: Open of /home/kaveh/miniconda3/envs/roadstyle/share/proj failed


,highway,name,oneway,maxspeed
0,tertiary,NaN,NaN,40
1,tertiary,NaN,NaN,40
2,secondary,Enköpingsvägen,yes,50


### A categorical example
We derive an importance class from the `highway` tag and give each value an explicit colour.

In [2]:
major  = {"motorway", "trunk", "primary", "motorway_link", "trunk_link", "primary_link"}
medium = {"secondary", "tertiary", "secondary_link", "tertiary_link"}
edges["importance"] = edges["highway"].apply(
    lambda h: "major" if h in major else "medium" if h in medium else "minor"
)
rs.render_edges(
    edges,
    color_by="importance",
    colors={"major": "#ef4444", "medium": "#f59e0b", "minor": "#64748b"},
    legend=True,
    compress=True,
)

### A numeric example
We compute each edge's length (in metres, via Sweden's projected CRS EPSG:3006) and map it to a colour ramp **and** a line-width ramp.

In [3]:
edges["length_m"] = edges.to_crs(3006).length.round(1)
rs.render_edges(
    edges,
    color_by="length_m",
    cmap="viridis",
    width_by=(1, 6),          # thinnest .. thickest pixel width across the value range
    legend=True,
    compress=True,
)

`vmin`/`vmax` clamp the colour range; `width_by=(min,max)` scales width with the value. Any [branca](https://python-visualization.github.io/branca/)/matplotlib colormap name works for `cmap`.